In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install geoip2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.3/101.3 kB 4.6 MB/s eta 0:00:00


In [ ]:
import community as community_louvain
import geoip2.database
import random
from math import radians, sin, cos, sqrt, atan2
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate  # Ensure you have the 'tabulate' library installed
from collections import deque
import geoip2.webservice
import numpy as np
import csv
import heapq
import pandas as pd
from collections import deque

In [ ]:
|import pandas as pd
import random
import statistics
import heapq
# Read the CSV file
# file_path = './GeoLite2-City-CSV_20231128/GeoLite2-City-Blocks-IPv4.csv'
file_path = '/content/drive/MyDrive/P2p/GeoLite2-City-Blocks-IPv4.csv'


# Load the data into a pandas DataFrame
data = pd.read_csv(file_path)

# Get the total number of rows
total_rows = data.shape[0]

# Select 10,000 random rows
if total_rows > 10000:
    random_indices = random.sample(range(total_rows), 100000)
    random_data = data.iloc[random_indices]
else:
    random_data = data  # If the file has less than 10000 rows, use all rows

# Store the randomly selected data into another CSV file
random_data.to_csv('random_10000_values.csv', index=False)


SyntaxError: invalid syntax (2215672944.py, line 1)

In [ ]:
import pandas as pd

# Specify the path to your CSV file
csv_file_path = '/content/drive/MyDrive/P2p/GeoLite2-City-Blocks-IPv4.csv'

# Use pandas to read the CSV file into a DataFrame
df = pd.read_csv(csv_file_path)

# Extract two random values from the "Network" column
random_values = df['network']

# Display the randomly selected values
print(random_values)


50000 nodes

In [ ]:

class IPAddressGraph:
    def __init__(self):
        self.G = nx.Graph()
        self.reader = geoip2.database.Reader('/content/drive/MyDrive/P2p/GeoLite2-City.mmdb')
        self.receiving_times = {}
        self.tmpran = []
        self.new_count_started=0
        self.no_of_messages_passed=0
        self.r_max=None

    def haversine(self, lat1, lon1, lat2, lon2):
        # Convert latitude and longitude from degrees to radians
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

        # Haversine formula
        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
        c = 2 * atan2(sqrt(a), sqrt(1 - a))

        # Radius of Earth in kilometers (change it to 3958.8 miles for miles)
        r = 6371.0

        # Calculate the distance
        distance = r * c

        return distance

    def get_location(self, ip_address):
        try:
            response = self.reader.city(ip_address)
            if response.location.latitude is not None and response.location.longitude is not None:
                return response.location.latitude, response.location.longitude
        except geoip2.errors.AddressNotFoundError:
            return None
    def get_distance(self, u, v):
        """
        Calculate the Haversine distance between two nodes u and v in the graph.
        If the calculated distance is zero (i.e., nodes have the same coordinates),
        a default distance of 4 km is used.
        If any latitude/longitude is missing, it uses the `get_location` method to retrieve the missing coordinates.
        """
        # Retrieve latitude and longitude from the node attributes
        lat1, lon1 = self.G.nodes[u]['pos'][0], self.G.nodes[u]['pos'][1]
        lat2, lon2 = self.G.nodes[v]['pos'][0], self.G.nodes[v]['pos'][1]

        # Check if any coordinates are missing, and use get_location to retrieve them if so
        if lat1 is None or lon1 is None:
            #print(f"Warning: Missing latitude/longitude for node {u}. Fetching from external source.")
            location = self.get_location(u)  # Use get_location for missing coordinates
            if location:
                lat1, lon1 = location
                if lat1 is None or lon1 is None:
                    print(f"Still Missing latitude/longitude for node {u}. Failed from get location")
            else:
                print(f"Warning: Could not retrieve location for node {u}. Returning default distance.")
                return 4.0  # Default distance if location cannot be found

        if lat2 is None or lon2 is None:
            #print(f"Warning: Missing latitude/longitude for node {v}. Fetching from external source.")
            location = self.get_location(v)  # Use get_location for missing coordinates
            if location:
                lat2, lon2 = location
                if lat2 is None or lon2 is None:
                    print(f"Still Missing latitude/longitude for node {v}. Failed from get location")
            else:
                print(f"Warning: Could not retrieve location for node {v}. Returning default distance.")
                return 4.0  # Default distance if location cannot be found

        # Calculate the distance using the Haversine formula
        calculated_distance = self.haversine(lat1, lon1, lat2, lon2)

        # If the calculated distance is zero (nodes have the same location), assign 4 km
        if calculated_distance == 0:
            #print(f"Nodes {u} and {v} have the same location. Assigning default distance of 4 km.")
            calculated_distance = 4.0  # Default distance in km

        return calculated_distance


    # G is a random graph
    def add_node_connect_random(self, ip, num_connections):
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Connect the new node to randomly selected existing nodes
                existing_nodes = list(self.G.nodes)
                if len(existing_nodes) >= num_connections:
                    random_nodes = random.sample(existing_nodes, num_connections)
                    for node in random_nodes:
                        # Use get_distance instead of haversine
                        distance = self.get_distance(ip, node)
                        self.G.add_edge(ip, node, weight=distance)


    # original code - connect to nearest nodes without any rmax.
    def add_node_connect_nearest(self, ip, out_degree):
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Connect the new node to the nearest existing nodes
                existing_nodes = list(self.G.nodes)
                if len(existing_nodes) >= out_degree:
                    # Sort the existing nodes by distance from the new node using get_distance
                    existing_nodes.sort(key=lambda x: self.get_distance(ip, x))  # Using get_distance here
                    nearest_nodes = existing_nodes[:out_degree]

                    # Add edges between the new node and the nearest nodes
                    for node in nearest_nodes:
                        distance = self.get_distance(ip, node)  # Using get_distance for the edge weight
                        self.G.add_edge(ip, node, weight=distance)


    def pdbca(self, ip, r_min, alpha, beta):
        # Ensure the new node is added if not already present
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Calculate distance to all existing nodes using get_distance
                existing_nodes = list(self.G.nodes)
                distances = []
                distance_dict = {}
                max_distance = 0

                for node in existing_nodes:
                    if node != ip:
                        distance = self.get_distance(ip, node)  # Use get_distance here instead of haversine
                        distances.append((node, distance))
                        distance_dict[node] = distance
                        max_distance = max(max_distance, distance)  # Update max_distance

                # If there aren't enough existing nodes to connect to, return early
                if len(existing_nodes) <= r_min:
                    return  # Not enough nodes to connect to

                # Initialize r_max if it's not already initialized
                if self.r_max is None:
                    self.r_max = 4 * r_min

                # Calculate scores based on distance and degree
                all_scores = []
                for node, distance in distances:
                    distance_score = distance / max_distance
                    degree = self.G.degree(node)
                    degree_score = degree / self.r_max
                    score = alpha * distance_score + beta * degree_score
                    all_scores.append((node, score))

                # Sort nodes by the calculated score
                all_scores.sort(key=lambda x: x[1])

                selected_nodes = []
                selected_node_set = set()

                # Select nodes based on the calculated score and degree constraints
                while len(selected_nodes) < r_min:
                    for node, score in all_scores:
                        if node not in selected_node_set and self.G.degree(node) <= self.r_max:
                            selected_nodes.append((node, score))
                            selected_node_set.add(node)
                            if len(selected_nodes) >= r_min:
                                break

                    if len(selected_nodes) < r_min:
                        self.r_max += 1
                        if self.r_max > len(existing_nodes):
                            break

                # Connect new node to selected nodes
                for node, score in selected_nodes:
                    self.G.add_edge(ip, node, weight=distance_dict[node])

                return self.r_max







    def draw_graph(self, title='IP Addresses Network Graph', output_file='network_graph.png'):
        self_loops = list(nx.selfloop_edges(self.G))
        self.G.remove_edges_from(self_loops)
        largest_degree_node = max(self.G.degree, key=lambda x: x[1])[0]
        largest_degree = self.G.degree[largest_degree_node]
        plt.figure(figsize=(10, 6))
        pos = nx.spring_layout(self.G, k=0.15, iterations=50)
        node_colors = ['red' if node == largest_degree_node else 'skyblue' for node in self.G.nodes]
        nx.draw_networkx_nodes(
            self.G, pos,
            node_color=node_colors,
            node_size=300,
            edgecolors='black'
        )
        nx.draw_networkx_edges(
            self.G, pos,
            width=1.5,
            alpha=0.7,
            edge_color='gray'
        )
        labels = {node: f'{degree}' for node, degree in self.G.degree()}
        nx.draw_networkx_labels(self.G, pos, labels, font_size=10, font_color='black')
        plt.title(title, fontsize=15)
        plt.axis('off')
        plt.savefig(title+"_new.png", format='png', bbox_inches='tight')
        plt.show()

    def calculate_distances_table(self):
        distances = []
        nodes = list(self.G.nodes)
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                node1, node2 = nodes[i], nodes[j]
                distance = self.haversine(
                    self.G.nodes[node1]['pos'][0], self.G.nodes[node1]['pos'][1],
                    self.G.nodes[node2]['pos'][0], self.G.nodes[node2]['pos'][1]
                )
                distances.append({'Node 1': node1, 'Node 2': node2, 'Distance (km)': distance})
        distances_df = pd.DataFrame(distances)
        print(tabulate(distances_df, headers='keys', tablefmt='pretty'))



    def random_n(self):
        all_nodes = list(self.G.nodes)
        rv=random.choice(all_nodes)
        if self.new_count_started:
            self.tmpran = [rv]
        return self.tmpran[0]

    def calculate_convergence_time(self, random_node):
            # Check if the graph has nodes
            if len(self.G.nodes) == 0:
                print("The graph doesn't contain any nodes.")
                return 0, 0  # Return zero for both convergence time and messages forwarded

            all_nodes = list(self.G.nodes)
            receive_times = {node: float('inf') for node in all_nodes}
            receive_times[random_node] = 0  # The starting node receives the message at time 0

            event_queue = [(0, random_node, None)]  # Priority queue for event-driven simulation
            heapq.heapify(event_queue)

            visited = set()  # Set to track visited nodes (nodes that have received the message)
            self.total_messages_forwarded = 0  # Reset the total messages forwarded counter

            while event_queue:
                current_time, current_node, sender = heapq.heappop(event_queue)

                if current_node in visited:
                    continue  # Skip if the node has already received the message

                visited.add(current_node)  # Mark the node as visited (it has received the message)

                for neighbor in self.G.neighbors(current_node):
                    if neighbor != sender:  # Avoid sending back to the sender
                        distance = self.G.edges[current_node, neighbor]['weight']
                        delay_ms = self.calculate_dynamic_delay(distance)  # Calculate delay based on distance
                        new_receive_time = current_time + delay_ms  # Calculate when the neighbor will receive the message

                        # Increment the total messages forwarded counter (this happens every time a message is forwarded)
                        self.total_messages_forwarded += 1

                        # Only forward the message if the neighbor has not received it sooner
                        if new_receive_time < receive_times[neighbor]:
                            receive_times[neighbor] = new_receive_time
                            heapq.heappush(event_queue, (new_receive_time, neighbor, current_node))

            # Calculate the convergence time (i.e., the max receive time of all reachable nodes)
            convergence_time = max(receive_times[node] for node in receive_times if receive_times[node] != float('inf'))

            print(f"Total messages forwarded: {self.total_messages_forwarded}")
            return convergence_time, self.total_messages_forwarded

    def calculate_dynamic_delay(self, distance):
        """
        Calculate delay based on distance. Adjust for realistic network behavior.
        """
        return (distance / 1000) * 4  # Example: 4 ms per km
    import random

    def calculate_average_convergence_time(self):
        # Randomly sample 10 nodes from the graph for testing
        random_nodes = random.sample(list(self.G.nodes), 30)

        # Lists to store convergence times and messages forwarded for each random node
        convergence_times = []
        total_messages = []

        # Iterate over the sampled random nodes
        for start_node in random_nodes:
            # Calculate the convergence time and messages forwarded for the current node
            convergence_time, messages = self.calculate_convergence_time(start_node)
            convergence_times.append(convergence_time)
            total_messages.append(messages)

        # Calculate the average convergence time
        average_convergence_time = sum(convergence_times) / len(convergence_times)
        average_messages_forwarded = sum(total_messages) / len(total_messages)

        return average_convergence_time, average_messages_forwarded
    def graph_out_degree_vs_convergence_time(self,out_degrees,convergence_times,message):
        # Plotting the graph for out_degree vs convergence_time
        plt.plot(out_degrees, convergence_times, marker='o')
        plt.xlabel('Out Degree')
        plt.ylabel('Convergence Time')
        plt.title(message)
        plt.grid(True)
        plt.show()

    import matplotlib.pyplot as plt
    # Function to save node degrees in CSV format
    def save_node_degrees_to_csv(self,graph, method_name, file_path):
        node_degrees = []
        # Iterate over all nodes in the graph
        for node in graph.nodes():
            # Get the degree of the node
            node_degree = graph.degree(node)
            node_degrees.append((node, node_degree))

        # Write node degrees to CSV file
        with open(file_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Node", "Degree"])  # Write header
            for node, degree in node_degrees:
                writer.writerow([node, degree])  # Write node and its degree

    def calculate_node_degrees(self, title, output_filename):
        degrees = dict(self.G.degree())
        node_degrees = list(degrees.values())

        # Count the occurrences of each degree
        degree_count = {}
        for degree in node_degrees:
            if degree in degree_count:
                degree_count[degree] += 1
            else:
                degree_count[degree] = 1
        # Writing the degree counts to a CSV file
        # Writing the degree counts to a CSV file
        with open(output_filename, mode='w', newline='') as file:
            writer = csv.writer(file)

            # Write header
            writer.writerow(['Degree', 'Count'])

            # Write the degree counts to the file
            for degree, count in degree_count.items():
                writer.writerow([degree, count])

        print(f"Degree counts saved to {output_filename}")






        return degree_count









# Usage example
ip_graph = IPAddressGraph()

node_counts = [50000]
out_degrees = [8]  # Different - rmin as 3.

# Initialize a dictionary to store degree information for each node count
degree_tables = {}
degree_tables_random={}
degree_tables_nearest_limit={}
degree_tables_optimal_75 = {}
degree_tables_optimal = {}
degree_tables_optimal_25 = {}
message_NNC=0
message_random=0
message_PDBCA_75=0
message_PDBCA_50=0
message_PDBCA_25=0
messages_passed_data = []
convergence_times = []
convergence_times_random=[]
convergence_times_nearest_limit=[]
convergence_times_optimal_75 = []
convergence_times_optimal = []
convergence_times_optimal_25 = []
r_max_values = []
import pandas as pd
import random
import statistics
import time
for count in node_counts:


    selected_ips = pd.read_csv('/content/drive/MyDrive/P2p/Final_Unique_IPs.csv')

    # Take a sample of 'count' number of IP addresses
    selected_ips = selected_ips.sample(count)
    ip_graph.new_count_started=1


    convergence_times_for_count = []
    convergence_times_for_count_random=[]
    convergence_times_for_count_nearest_limit=[]
    convergence_times_for_count_optimal_75 = []
    convergence_times_for_count_optimal = []
    convergence_times_for_count_optimal_25 = []
    messages_passed_per_count = []

    # Add nodes to the graph using the module and connect it to nearest
    res = []
    resran = []
    res_nearest_limit = []
    res_optimal_75 = []
    res_optimal = []
    res_optimal_25 = []

    total_degrees_nnc = 0
    total_nodes_nnc = 0

    total_degrees_random = 0
    total_nodes_random = 0

    total_degrees_dbnnc = 0
    total_nodes_dbnnc = 0

    total_degrees_optimal_75 = 0
    total_degrees_optimal = 0
    total_degrees_optimal_25 = 0

    total_nodes_optimal_75 = 0
    total_nodes_optimal = 0
    total_nodes_optimal_25 = 0
    start_time = time.time()
    last_interval_time = start_time

    for degree in out_degrees:
        resit = []
        randresit = []
        randresitlimit = []
        res_opt_75 = []
        res_opt = []
        res_opt_25 = []
        ip_graph.r_max=None
        for _ in range(1):
            ip_graph.G.clear()  # Clear the graph for each out_degree
            c=0
            """
            for ip in selected_ips['IP_Address']:
                ip_graph.add_node_connect_nearest(ip,degree)
                c=c+1
                if(c%2000 == 0):
                  print(f"{c} number of nodes done")
                if c % 10000 == 0:
                  current_time = time.time()

                  # Calculate how long the last 10k nodes took
                  interval_duration = current_time - last_interval_time
                  # Calculate total time since the beginning
                  total_elapsed = current_time - start_time

                  print(f"--- Progress: {c} nodes ---")
                  print(f"Time for this 10k block: {interval_duration/60:.2f} minutes")
                  print(f"Total time elapsed: {total_elapsed/60:.2f} minutes")
                  print("-" * 25)

                  # Reset the interval marker
                  last_interval_time = current_time
                  print(f"Number of nodes: {ip_graph.G.number_of_nodes()}")
            # ip_graph.draw_graph("IP address graph with nearest neighbours")
            #ip_graph.draw_graph("NNC Graph")
            print("Nearest neigbour connection")
            print(f"number of nodes{ip_graph.G.number_of_nodes()} ")
            # Save node degrees for NNC method

            ip_graph.save_node_degrees_to_csv(ip_graph.G, "NNC", "node_degrees_NNC_{degree}.csv")
            degree_info= ip_graph.calculate_node_degrees(f"degree distribution in nearest graph of degree {degree} node count {count}","NNC_50000_4_rmin.csv")
            # Store degree information in the dictionary
            degree_tables[count] = degree_info
            # ip_graph.calculate_node_degrees(degree)
            # convergence_time = ip_graph.calculate_convergence_time()
            convergence_time,message_NNC = ip_graph.calculate_average_convergence_time()
            print(f"NCC Average Convergence Time over 30 iterations: {convergence_time}")
            # convergence_time=ip_graph.calculate_convergence_time_new()

            # ip_graph.draw_graph("NNC Graph")

            resit.append(convergence_time)
            # convergence_times_for_count.append(convergence_time)
            ip_graph.new_count_started=0
            # Update total degrees and nodes for NNC
            total_degrees_nnc += sum(degree_info.keys())
            total_nodes_nnc += len(degree_info.keys())
            print("Total sum degree nnc:",total_degrees_nnc)
            print("Total sum nodes nnc:",total_nodes_nnc)
            total_edges = ip_graph.G.number_of_edges()
            print(f"Total number of edges in nnc: {total_edges}")

            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for NNC: {total_degree}")
            print(f"Average degree for NNC: {average_degree}")
            # convergence_times.append(convergence_time)
            # loop for random connection
            """
            ip_graph.G.clear()
            c=0
            for ip in selected_ips['IP_Address']:
                ip_graph.add_node_connect_random(ip,degree)
                c=c+1
                if(c%2000 == 0):
                  print(f"{c} number of nodes done")
                if c % 10000 == 0:
                  current_time = time.time()

                  # Calculate how long the last 10k nodes took
                  interval_duration = current_time - last_interval_time
                  # Calculate total time since the beginning
                  total_elapsed = current_time - start_time

                  print(f"--- Progress: {c} nodes ---")
                  print(f"Time for this 10k block: {interval_duration/60:.2f} minutes")
                  print(f"Total time elapsed: {total_elapsed/60:.2f} minutes")
                  print("-" * 25)

                  # Reset the interval marker
                  last_interval_time = current_time
            print("Random neigbour connection")
            print(f"number of nodes{ip_graph.G.number_of_nodes()} ")
            # Save node degrees for Random method
            ip_graph.save_node_degrees_to_csv(ip_graph.G, "Random", f"node_degrees_Random_{degree}.csv")
            degree_info_random=ip_graph.calculate_node_degrees(f"degree distribution in random graph of degree {degree} node count {count}","Random_50000_4_rmin.csv")
            degree_tables_random[count]=degree_info_random
             # ----------------------------------------
            #Degree Range Distribution Calculation
            # ----------------------------------------

            degree_ranges = {
                "5-10": 0,
                "11-20": 0,
                "21-50": 0,
                "51-100": 0,
                "101-500": 0,
                ">500": 0
            }

            # Count nodes in each degree range
            for node, deg in ip_graph.G.degree():
                if 5 <= deg <= 10:
                    degree_ranges["5-10"] += 1
                elif 11 <= deg <= 20:
                    degree_ranges["11-20"] += 1
                elif 21 <= deg <= 50:
                    degree_ranges["21-50"] += 1
                elif 51 <= deg <= 100:
                    degree_ranges["51-100"] += 1
                elif 101 <= deg <= 500:
                    degree_ranges["101-500"] += 1
                elif deg > 500:
                    degree_ranges[">500"] += 1

            # ----------------------------------------
            # Display Results (TwinPeer column)
            # ----------------------------------------

            print("\nNode Degree Range\tTwinPeer")
            for k, v in degree_ranges.items():
                print(f"{k}\t\t{v}")
            # convergence_time_random=ip_graph.calculate_convergence_time()
            convergence_time_random,message_random = ip_graph.calculate_average_convergence_time()
            print(f"Random Average Convergence Time over 30 iterations: {convergence_time_random}")
            #ip_graph.draw_graph("Random Graph")
            #print(f"total message forwarded calculated by degree random{ip_graph.total_messages_forwarded()}")
                # convergence_times_for_count_random.append(convergence_time_random)
            randresit.append(convergence_time_random)
            # Update total degrees and nodes for Random
            total_degrees_random += sum(degree_info_random.keys())
            total_nodes_random += len(degree_info_random.keys())
            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for Random: {total_degree}")
            print(f"Average degree for Random: {average_degree}")
            """
            ip_graph.G.clear()
            alpha=0.5
            beta=0.5
            for ip in selected_ips['network']:
                r_max_optimal = ip_graph.pdbca(ip, degree,alpha,beta)
                r_max_values.append(r_max_optimal)
            print("PDBCA connection with alpha 0.5 and beta 0.5")
            # Save node degrees for PDBCA method
            ip_graph.save_node_degrees_to_csv(ip_graph.G, "PDBCA", f"node_degrees_PDBCA_{degree}.csv")
            print(f"r-max value: {ip_graph.r_max}")
            print(f"Number of nodes: {ip_graph.G.number_of_nodes()}")
            degree_info_optimal = ip_graph.calculate_node_degrees(f"Degree distribution in PDBCA graph of degree {degree} node count {count} with alpha 0.5 and beta 0.5","PDBCA_5000_4_rmin_alpha_50.csv")
            degree_tables_optimal[count] = degree_info_optimal
            convergence_time_optimal,message_PDBCA_50 = ip_graph.calculate_average_convergence_time()
            print(f"PDBCA Average Convergence Time over 30 iterations: {convergence_time_optimal}")
            #ip_graph.draw_graph("PDBCA Graph")
            # Create a bar chart for degree distribution
            degrees = list(degree_info_optimal.keys())
            counts = list(degree_info_optimal.values())

            plt.figure(figsize=(10, 6))  # Set the size of the figure
            plt.bar(degrees, counts, color='blue')

            plt.xlabel('Degree')
            plt.ylabel('Number of Nodes')
            plt.title(f'Degree Distribution in PDBCA Graph of Degree {degree} Node Count {count}')
            plt.grid(True)

            # Save the bar chart as an image file
            plt.savefig('/content/drive/MyDrive/P2p/p2p_10000/degree_distribution_PDBCA_alpha_50.png')
            plt.close()  # Close the plot to prevent it from displaying
          #  print(f"Total messages forwarded calculated by PDBCA: {ip_graph.total_messages_forwarded()}")
            res_opt.append(convergence_time_optimal)
            # Calculate the sum of degrees and the number of nodes
            # Update total degrees and nodes for Optimal DBNNC
            total_degrees_optimal += sum(degree_info_optimal.keys())

            total_nodes_optimal += len(degree_info_optimal.keys())


            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for PDBCA: {total_degree}")
            print(f"Average degree for PDBCA: {average_degree}")

            """




        #avgresit = statistics.mean(resit)
        avgrandresit = statistics.mean(randresit)
        #avgnearestlimit = statistics.mean(res_nearest_limit)
       # avgoptimal_75 = statistics.mean(res_opt_75)
        #avgoptimal = statistics.mean(res_opt)
        #avgoptimal_25 = statistics.mean(res_opt_25)

        #convergence_times_for_count.append(avgresit)
        convergence_times_for_count_random.append(avgrandresit)
       # convergence_times_for_count_nearest_limit.append(avgnearestlimit)
        #convergence_times_for_count_optimal_75.append(avgoptimal_75)
        #convergence_times_for_count_optimal.append(avgoptimal)
        #convergence_times_for_count_optimal_25.append(avgoptimal_25)

    messages_passed_data.append(messages_passed_per_count)


    #convergence_times.append(convergence_times_for_count)
    convergence_times_random.append(convergence_times_for_count_random)
  #  convergence_times_nearest_limit.append(convergence_times_for_count_nearest_limit)
    #convergence_times_optimal_75.append(convergence_times_for_count_optimal_75)
    #convergence_times_optimal.append(convergence_times_for_count_optimal)
   # convergence_times_optimal_25.append(convergence_times_for_count_optimal_25)
# Calculate and print the average degree distribution for each connection type
"""
if total_nodes_nnc > 0:
    average_degree_nnc = total_degrees_nnc / total_nodes_nnc
    print(f"Average Degree Distribution for NNC: {average_degree_nnc}")
else:
    print("No nodes found in the NNC graph.")
"""
if total_nodes_random > 0:
    average_degree_random = total_degrees_random / total_nodes_random
    print(f"Average Degree Distribution for Random: {average_degree_random}")
else:
    print("No nodes found in the Random graph.")



"""
if total_nodes_optimal > 0:
    average_degree_optimal = total_degrees_optimal / total_nodes_optimal
    print(f"Average Degree Distribution for PDBCA with alpha 0.5 and beta 0.5: {average_degree_optimal}")
else:
    print("No nodes found in the Degree balanced NNC graph.")
"""

# Print final results
#print("Convergence Times for NNC:", convergence_times)
print("Convergence Times for Random Connection:", convergence_times_random)
#print("Convergence Times for Nearest Neighbor with Degree Limit:", convergence_times_nearest_limit)
#print("Convergence Times for PDBCA with alpha 0.75 and beta 0.25:", convergence_times_optimal_75)
#print("Convergence Times for  PDBCA with alpha 0.5 and beta 0.5:", convergence_times_optimal)
#print("Convergence Times for  PDBCA with alpha 0.25 and beta 0.75:", convergence_times_optimal_25)
print("################################################################")
#print("Total Messages forwarded for NNC:", message_NNC)
print("Total Messages forwarded for Random Connection:", message_random)
#print("Convergence Times for Nearest Neighbor with Degree Limit:", convergence_times_nearest_limit)
#print("Total Messages forwarded for PDBCA with alpha 0.75 and beta 0.25:", message_PDBCA_75)
#print("Total Messages forwarded  for  PDBCA with alpha 0.5 and beta 0.5:", message_PDBCA_50)
#print("Total Messages forwarded  for  PDBCA with alpha 0.25 and beta 0.75:", message_PDBCA_25)












2000 number of nodes done
4000 number of nodes done
6000 number of nodes done
8000 number of nodes done
10000 number of nodes done
--- Progress: 10000 nodes ---
Time for this 10k block: 0.07 minutes
Total time elapsed: 0.07 minutes
-------------------------
12000 number of nodes done
14000 number of nodes done
16000 number of nodes done
18000 number of nodes done
20000 number of nodes done
--- Progress: 20000 nodes ---
Time for this 10k block: 0.06 minutes
Total time elapsed: 0.13 minutes
-------------------------
22000 number of nodes done
24000 number of nodes done
26000 number of nodes done
28000 number of nodes done
30000 number of nodes done
--- Progress: 30000 nodes ---
Time for this 10k block: 0.08 minutes
Total time elapsed: 0.21 minutes
-------------------------
32000 number of nodes done
34000 number of nodes done
36000 number of nodes done
38000 number of nodes done
40000 number of nodes done
--- Progress: 40000 nodes ---
Time for this 10k block: 0.14 minutes
Total time elap

#30000 Nodes

In [ ]:

class IPAddressGraph:
    def __init__(self):
        self.G = nx.Graph()
        self.reader = geoip2.database.Reader('/content/drive/MyDrive/P2p/GeoLite2-City.mmdb')
        self.receiving_times = {}
        self.tmpran = []
        self.new_count_started=0
        self.no_of_messages_passed=0
        self.r_max=None

    def haversine(self, lat1, lon1, lat2, lon2):
        # Convert latitude and longitude from degrees to radians
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

        # Haversine formula
        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
        c = 2 * atan2(sqrt(a), sqrt(1 - a))

        # Radius of Earth in kilometers (change it to 3958.8 miles for miles)
        r = 6371.0

        # Calculate the distance
        distance = r * c

        return distance

    def get_location(self, ip_address):
        try:
            response = self.reader.city(ip_address)
            if response.location.latitude is not None and response.location.longitude is not None:
                return response.location.latitude, response.location.longitude
        except geoip2.errors.AddressNotFoundError:
            return None
    def get_distance(self, u, v):
        """
        Calculate the Haversine distance between two nodes u and v in the graph.
        If the calculated distance is zero (i.e., nodes have the same coordinates),
        a default distance of 4 km is used.
        If any latitude/longitude is missing, it uses the `get_location` method to retrieve the missing coordinates.
        """
        # Retrieve latitude and longitude from the node attributes
        lat1, lon1 = self.G.nodes[u]['pos'][0], self.G.nodes[u]['pos'][1]
        lat2, lon2 = self.G.nodes[v]['pos'][0], self.G.nodes[v]['pos'][1]

        # Check if any coordinates are missing, and use get_location to retrieve them if so
        if lat1 is None or lon1 is None:
            #print(f"Warning: Missing latitude/longitude for node {u}. Fetching from external source.")
            location = self.get_location(u)  # Use get_location for missing coordinates
            if location:
                lat1, lon1 = location
                if lat1 is None or lon1 is None:
                    print(f"Still Missing latitude/longitude for node {u}. Failed from get location")
            else:
                print(f"Warning: Could not retrieve location for node {u}. Returning default distance.")
                return 4.0  # Default distance if location cannot be found

        if lat2 is None or lon2 is None:
            #print(f"Warning: Missing latitude/longitude for node {v}. Fetching from external source.")
            location = self.get_location(v)  # Use get_location for missing coordinates
            if location:
                lat2, lon2 = location
                if lat2 is None or lon2 is None:
                    print(f"Still Missing latitude/longitude for node {v}. Failed from get location")
            else:
                print(f"Warning: Could not retrieve location for node {v}. Returning default distance.")
                return 4.0  # Default distance if location cannot be found

        # Calculate the distance using the Haversine formula
        calculated_distance = self.haversine(lat1, lon1, lat2, lon2)

        # If the calculated distance is zero (nodes have the same location), assign 4 km
        if calculated_distance == 0:
            #print(f"Nodes {u} and {v} have the same location. Assigning default distance of 4 km.")
            calculated_distance = 4.0  # Default distance in km

        return calculated_distance


    # G is a random graph
    def add_node_connect_random(self, ip, num_connections):
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Connect the new node to randomly selected existing nodes
                existing_nodes = list(self.G.nodes)
                if len(existing_nodes) >= num_connections:
                    random_nodes = random.sample(existing_nodes, num_connections)
                    for node in random_nodes:
                        # Use get_distance instead of haversine
                        distance = self.get_distance(ip, node)
                        self.G.add_edge(ip, node, weight=distance)


    # original code - connect to nearest nodes without any rmax.
    def add_node_connect_nearest(self, ip, out_degree):
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Connect the new node to the nearest existing nodes
                existing_nodes = list(self.G.nodes)
                if len(existing_nodes) >= out_degree:
                    # Sort the existing nodes by distance from the new node using get_distance
                    existing_nodes.sort(key=lambda x: self.get_distance(ip, x))  # Using get_distance here
                    nearest_nodes = existing_nodes[:out_degree]

                    # Add edges between the new node and the nearest nodes
                    for node in nearest_nodes:
                        distance = self.get_distance(ip, node)  # Using get_distance for the edge weight
                        self.G.add_edge(ip, node, weight=distance)


    def pdbca(self, ip, r_min, alpha, beta):
        # Ensure the new node is added if not already present
        if ip not in self.G.nodes:
            location = self.get_location(ip)
            if location:
                self.G.add_node(ip, pos=location)

                # Calculate distance to all existing nodes using get_distance
                existing_nodes = list(self.G.nodes)
                distances = []
                distance_dict = {}
                max_distance = 0

                for node in existing_nodes:
                    if node != ip:
                        distance = self.get_distance(ip, node)  # Use get_distance here instead of haversine
                        distances.append((node, distance))
                        distance_dict[node] = distance
                        max_distance = max(max_distance, distance)  # Update max_distance

                # If there aren't enough existing nodes to connect to, return early
                if len(existing_nodes) <= r_min:
                    return  # Not enough nodes to connect to

                # Initialize r_max if it's not already initialized
                if self.r_max is None:
                    self.r_max = 4 * r_min

                # Calculate scores based on distance and degree
                all_scores = []
                for node, distance in distances:
                    distance_score = distance / max_distance
                    degree = self.G.degree(node)
                    degree_score = degree / self.r_max
                    score = alpha * distance_score + beta * degree_score
                    all_scores.append((node, score))

                # Sort nodes by the calculated score
                all_scores.sort(key=lambda x: x[1])

                selected_nodes = []
                selected_node_set = set()

                # Select nodes based on the calculated score and degree constraints
                while len(selected_nodes) < r_min:
                    for node, score in all_scores:
                        if node not in selected_node_set and self.G.degree(node) <= self.r_max:
                            selected_nodes.append((node, score))
                            selected_node_set.add(node)
                            if len(selected_nodes) >= r_min:
                                break

                    if len(selected_nodes) < r_min:
                        self.r_max += 1
                        if self.r_max > len(existing_nodes):
                            break

                # Connect new node to selected nodes
                for node, score in selected_nodes:
                    self.G.add_edge(ip, node, weight=distance_dict[node])

                return self.r_max







    def draw_graph(self, title='IP Addresses Network Graph', output_file='network_graph.png'):
        self_loops = list(nx.selfloop_edges(self.G))
        self.G.remove_edges_from(self_loops)
        largest_degree_node = max(self.G.degree, key=lambda x: x[1])[0]
        largest_degree = self.G.degree[largest_degree_node]
        plt.figure(figsize=(10, 6))
        pos = nx.spring_layout(self.G, k=0.15, iterations=50)
        node_colors = ['red' if node == largest_degree_node else 'skyblue' for node in self.G.nodes]
        nx.draw_networkx_nodes(
            self.G, pos,
            node_color=node_colors,
            node_size=300,
            edgecolors='black'
        )
        nx.draw_networkx_edges(
            self.G, pos,
            width=1.5,
            alpha=0.7,
            edge_color='gray'
        )
        labels = {node: f'{degree}' for node, degree in self.G.degree()}
        nx.draw_networkx_labels(self.G, pos, labels, font_size=10, font_color='black')
        plt.title(title, fontsize=15)
        plt.axis('off')
        plt.savefig(title+"_new.png", format='png', bbox_inches='tight')
        plt.show()

    def calculate_distances_table(self):
        distances = []
        nodes = list(self.G.nodes)
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                node1, node2 = nodes[i], nodes[j]
                distance = self.haversine(
                    self.G.nodes[node1]['pos'][0], self.G.nodes[node1]['pos'][1],
                    self.G.nodes[node2]['pos'][0], self.G.nodes[node2]['pos'][1]
                )
                distances.append({'Node 1': node1, 'Node 2': node2, 'Distance (km)': distance})
        distances_df = pd.DataFrame(distances)
        print(tabulate(distances_df, headers='keys', tablefmt='pretty'))



    def random_n(self):
        all_nodes = list(self.G.nodes)
        rv=random.choice(all_nodes)
        if self.new_count_started:
            self.tmpran = [rv]
        return self.tmpran[0]

    def calculate_convergence_time(self, random_node):
            # Check if the graph has nodes
            if len(self.G.nodes) == 0:
                print("The graph doesn't contain any nodes.")
                return 0, 0  # Return zero for both convergence time and messages forwarded

            all_nodes = list(self.G.nodes)
            receive_times = {node: float('inf') for node in all_nodes}
            receive_times[random_node] = 0  # The starting node receives the message at time 0

            event_queue = [(0, random_node, None)]  # Priority queue for event-driven simulation
            heapq.heapify(event_queue)

            visited = set()  # Set to track visited nodes (nodes that have received the message)
            self.total_messages_forwarded = 0  # Reset the total messages forwarded counter

            while event_queue:
                current_time, current_node, sender = heapq.heappop(event_queue)

                if current_node in visited:
                    continue  # Skip if the node has already received the message

                visited.add(current_node)  # Mark the node as visited (it has received the message)

                for neighbor in self.G.neighbors(current_node):
                    if neighbor != sender:  # Avoid sending back to the sender
                        distance = self.G.edges[current_node, neighbor]['weight']
                        delay_ms = self.calculate_dynamic_delay(distance)  # Calculate delay based on distance
                        new_receive_time = current_time + delay_ms  # Calculate when the neighbor will receive the message

                        # Increment the total messages forwarded counter (this happens every time a message is forwarded)
                        self.total_messages_forwarded += 1

                        # Only forward the message if the neighbor has not received it sooner
                        if new_receive_time < receive_times[neighbor]:
                            receive_times[neighbor] = new_receive_time
                            heapq.heappush(event_queue, (new_receive_time, neighbor, current_node))

            # Calculate the convergence time (i.e., the max receive time of all reachable nodes)
            convergence_time = max(receive_times[node] for node in receive_times if receive_times[node] != float('inf'))

            print(f"Total messages forwarded: {self.total_messages_forwarded}")
            return convergence_time, self.total_messages_forwarded

    def calculate_dynamic_delay(self, distance):
        """
        Calculate delay based on distance. Adjust for realistic network behavior.
        """
        return (distance / 1000) * 4  # Example: 4 ms per km
    import random

    def calculate_average_convergence_time(self):
        # Randomly sample 10 nodes from the graph for testing
        random_nodes = random.sample(list(self.G.nodes), 30)

        # Lists to store convergence times and messages forwarded for each random node
        convergence_times = []
        total_messages = []

        # Iterate over the sampled random nodes
        for start_node in random_nodes:
            # Calculate the convergence time and messages forwarded for the current node
            convergence_time, messages = self.calculate_convergence_time(start_node)
            convergence_times.append(convergence_time)
            total_messages.append(messages)

        # Calculate the average convergence time
        average_convergence_time = sum(convergence_times) / len(convergence_times)
        average_messages_forwarded = sum(total_messages) / len(total_messages)

        return average_convergence_time, average_messages_forwarded
    def graph_out_degree_vs_convergence_time(self,out_degrees,convergence_times,message):
        # Plotting the graph for out_degree vs convergence_time
        plt.plot(out_degrees, convergence_times, marker='o')
        plt.xlabel('Out Degree')
        plt.ylabel('Convergence Time')
        plt.title(message)
        plt.grid(True)
        plt.show()

    import matplotlib.pyplot as plt
    # Function to save node degrees in CSV format
    def save_node_degrees_to_csv(self,graph, method_name, file_path):
        node_degrees = []
        # Iterate over all nodes in the graph
        for node in graph.nodes():
            # Get the degree of the node
            node_degree = graph.degree(node)
            node_degrees.append((node, node_degree))

        # Write node degrees to CSV file
        with open(file_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Node", "Degree"])  # Write header
            for node, degree in node_degrees:
                writer.writerow([node, degree])  # Write node and its degree

    def calculate_node_degrees(self, title, output_filename):
        degrees = dict(self.G.degree())
        node_degrees = list(degrees.values())

        # Count the occurrences of each degree
        degree_count = {}
        for degree in node_degrees:
            if degree in degree_count:
                degree_count[degree] += 1
            else:
                degree_count[degree] = 1
        # Writing the degree counts to a CSV file
        # Writing the degree counts to a CSV file
        with open(output_filename, mode='w', newline='') as file:
            writer = csv.writer(file)

            # Write header
            writer.writerow(['Degree', 'Count'])

            # Write the degree counts to the file
            for degree, count in degree_count.items():
                writer.writerow([degree, count])

        print(f"Degree counts saved to {output_filename}")






        return degree_count









# Usage example
ip_graph = IPAddressGraph()

node_counts = [30000]
out_degrees = [8]  # Different - rmin as 3.

# Initialize a dictionary to store degree information for each node count
degree_tables = {}
degree_tables_random={}
degree_tables_nearest_limit={}
degree_tables_optimal_75 = {}
degree_tables_optimal = {}
degree_tables_optimal_25 = {}
message_NNC=0
message_random=0
message_PDBCA_75=0
message_PDBCA_50=0
message_PDBCA_25=0
messages_passed_data = []
convergence_times = []
convergence_times_random=[]
convergence_times_nearest_limit=[]
convergence_times_optimal_75 = []
convergence_times_optimal = []
convergence_times_optimal_25 = []
r_max_values = []
import pandas as pd
import random
import statistics
import time
for count in node_counts:


    selected_ips = pd.read_csv('/content/drive/MyDrive/P2p/Final_Unique_IPs.csv')

    # Take a sample of 'count' number of IP addresses
    selected_ips = selected_ips.sample(count)
    ip_graph.new_count_started=1


    convergence_times_for_count = []
    convergence_times_for_count_random=[]
    convergence_times_for_count_nearest_limit=[]
    convergence_times_for_count_optimal_75 = []
    convergence_times_for_count_optimal = []
    convergence_times_for_count_optimal_25 = []
    messages_passed_per_count = []

    # Add nodes to the graph using the module and connect it to nearest
    res = []
    resran = []
    res_nearest_limit = []
    res_optimal_75 = []
    res_optimal = []
    res_optimal_25 = []

    total_degrees_nnc = 0
    total_nodes_nnc = 0

    total_degrees_random = 0
    total_nodes_random = 0

    total_degrees_dbnnc = 0
    total_nodes_dbnnc = 0

    total_degrees_optimal_75 = 0
    total_degrees_optimal = 0
    total_degrees_optimal_25 = 0

    total_nodes_optimal_75 = 0
    total_nodes_optimal = 0
    total_nodes_optimal_25 = 0
    start_time = time.time()
    last_interval_time = start_time

    for degree in out_degrees:
        resit = []
        randresit = []
        randresitlimit = []
        res_opt_75 = []
        res_opt = []
        res_opt_25 = []
        ip_graph.r_max=None
        for _ in range(1):
            ip_graph.G.clear()  # Clear the graph for each out_degree
            c=0
            """
            for ip in selected_ips['IP_Address']:
                ip_graph.add_node_connect_nearest(ip,degree)
                c=c+1
                if(c%2000 == 0):
                  print(f"{c} number of nodes done")
                if c % 10000 == 0:
                  current_time = time.time()

                  # Calculate how long the last 10k nodes took
                  interval_duration = current_time - last_interval_time
                  # Calculate total time since the beginning
                  total_elapsed = current_time - start_time

                  print(f"--- Progress: {c} nodes ---")
                  print(f"Time for this 10k block: {interval_duration/60:.2f} minutes")
                  print(f"Total time elapsed: {total_elapsed/60:.2f} minutes")
                  print("-" * 25)

                  # Reset the interval marker
                  last_interval_time = current_time
                  print(f"Number of nodes: {ip_graph.G.number_of_nodes()}")
            # ip_graph.draw_graph("IP address graph with nearest neighbours")
            #ip_graph.draw_graph("NNC Graph")
            print("Nearest neigbour connection")
            print(f"number of nodes{ip_graph.G.number_of_nodes()} ")
            # Save node degrees for NNC method

            ip_graph.save_node_degrees_to_csv(ip_graph.G, "NNC", "node_degrees_NNC_{degree}.csv")
            degree_info= ip_graph.calculate_node_degrees(f"degree distribution in nearest graph of degree {degree} node count {count}","NNC_50000_4_rmin.csv")
            # Store degree information in the dictionary
            degree_tables[count] = degree_info
            # ip_graph.calculate_node_degrees(degree)
            # convergence_time = ip_graph.calculate_convergence_time()
            convergence_time,message_NNC = ip_graph.calculate_average_convergence_time()
            print(f"NCC Average Convergence Time over 30 iterations: {convergence_time}")
            # convergence_time=ip_graph.calculate_convergence_time_new()

            # ip_graph.draw_graph("NNC Graph")

            resit.append(convergence_time)
            # convergence_times_for_count.append(convergence_time)
            ip_graph.new_count_started=0
            # Update total degrees and nodes for NNC
            total_degrees_nnc += sum(degree_info.keys())
            total_nodes_nnc += len(degree_info.keys())
            print("Total sum degree nnc:",total_degrees_nnc)
            print("Total sum nodes nnc:",total_nodes_nnc)
            total_edges = ip_graph.G.number_of_edges()
            print(f"Total number of edges in nnc: {total_edges}")

            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for NNC: {total_degree}")
            print(f"Average degree for NNC: {average_degree}")
            # convergence_times.append(convergence_time)
            # loop for random connection
            """
            ip_graph.G.clear()
            c=0
            for ip in selected_ips['IP_Address']:
                ip_graph.add_node_connect_random(ip,degree)
                c=c+1
                if(c%2000 == 0):
                  print(f"{c} number of nodes done")
                if c % 10000 == 0:
                  current_time = time.time()

                  # Calculate how long the last 10k nodes took
                  interval_duration = current_time - last_interval_time
                  # Calculate total time since the beginning
                  total_elapsed = current_time - start_time

                  print(f"--- Progress: {c} nodes ---")
                  print(f"Time for this 10k block: {interval_duration/60:.2f} minutes")
                  print(f"Total time elapsed: {total_elapsed/60:.2f} minutes")
                  print("-" * 25)

                  # Reset the interval marker
                  last_interval_time = current_time
            print("Random neigbour connection")
            print(f"number of nodes{ip_graph.G.number_of_nodes()} ")
            # Save node degrees for Random method
            ip_graph.save_node_degrees_to_csv(ip_graph.G, "Random", f"node_degrees_Random_{degree}.csv")
            degree_info_random=ip_graph.calculate_node_degrees(f"degree distribution in random graph of degree {degree} node count {count}","Random_50000_4_rmin.csv")
            degree_tables_random[count]=degree_info_random

            # convergence_time_random=ip_graph.calculate_convergence_time()
            convergence_time_random,message_random = ip_graph.calculate_average_convergence_time()
            print(f"Random Average Convergence Time over 30 iterations: {convergence_time_random}")
            #ip_graph.draw_graph("Random Graph")
            #print(f"total message forwarded calculated by degree random{ip_graph.total_messages_forwarded()}")
                # convergence_times_for_count_random.append(convergence_time_random)
            randresit.append(convergence_time_random)
            # Update total degrees and nodes for Random
            total_degrees_random += sum(degree_info_random.keys())
            total_nodes_random += len(degree_info_random.keys())
            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for Random: {total_degree}")
            print(f"Average degree for Random: {average_degree}")
            """
            ip_graph.G.clear()
            alpha=0.5
            beta=0.5
            for ip in selected_ips['network']:
                r_max_optimal = ip_graph.pdbca(ip, degree,alpha,beta)
                r_max_values.append(r_max_optimal)
            print("PDBCA connection with alpha 0.5 and beta 0.5")
            # Save node degrees for PDBCA method
            ip_graph.save_node_degrees_to_csv(ip_graph.G, "PDBCA", f"node_degrees_PDBCA_{degree}.csv")
            print(f"r-max value: {ip_graph.r_max}")
            print(f"Number of nodes: {ip_graph.G.number_of_nodes()}")
            degree_info_optimal = ip_graph.calculate_node_degrees(f"Degree distribution in PDBCA graph of degree {degree} node count {count} with alpha 0.5 and beta 0.5","PDBCA_5000_4_rmin_alpha_50.csv")
            degree_tables_optimal[count] = degree_info_optimal
            convergence_time_optimal,message_PDBCA_50 = ip_graph.calculate_average_convergence_time()
            print(f"PDBCA Average Convergence Time over 30 iterations: {convergence_time_optimal}")
            #ip_graph.draw_graph("PDBCA Graph")
            # Create a bar chart for degree distribution
            degrees = list(degree_info_optimal.keys())
            counts = list(degree_info_optimal.values())

            plt.figure(figsize=(10, 6))  # Set the size of the figure
            plt.bar(degrees, counts, color='blue')

            plt.xlabel('Degree')
            plt.ylabel('Number of Nodes')
            plt.title(f'Degree Distribution in PDBCA Graph of Degree {degree} Node Count {count}')
            plt.grid(True)

            # Save the bar chart as an image file
            plt.savefig('/content/drive/MyDrive/P2p/p2p_10000/degree_distribution_PDBCA_alpha_50.png')
            plt.close()  # Close the plot to prevent it from displaying
          #  print(f"Total messages forwarded calculated by PDBCA: {ip_graph.total_messages_forwarded()}")
            res_opt.append(convergence_time_optimal)
            # Calculate the sum of degrees and the number of nodes
            # Update total degrees and nodes for Optimal DBNNC
            total_degrees_optimal += sum(degree_info_optimal.keys())

            total_nodes_optimal += len(degree_info_optimal.keys())


            total_degree = sum(dict(ip_graph.G.degree()).values())
            average_degree = total_degree / len(ip_graph.G.nodes())
            print(f"Sum of all degrees for PDBCA: {total_degree}")
            print(f"Average degree for PDBCA: {average_degree}")

            """




        #avgresit = statistics.mean(resit)
        avgrandresit = statistics.mean(randresit)
        #avgnearestlimit = statistics.mean(res_nearest_limit)
       # avgoptimal_75 = statistics.mean(res_opt_75)
        #avgoptimal = statistics.mean(res_opt)
        #avgoptimal_25 = statistics.mean(res_opt_25)

        #convergence_times_for_count.append(avgresit)
        convergence_times_for_count_random.append(avgrandresit)
       # convergence_times_for_count_nearest_limit.append(avgnearestlimit)
        #convergence_times_for_count_optimal_75.append(avgoptimal_75)
        #convergence_times_for_count_optimal.append(avgoptimal)
        #convergence_times_for_count_optimal_25.append(avgoptimal_25)

    messages_passed_data.append(messages_passed_per_count)


    #convergence_times.append(convergence_times_for_count)
    convergence_times_random.append(convergence_times_for_count_random)
  #  convergence_times_nearest_limit.append(convergence_times_for_count_nearest_limit)
    #convergence_times_optimal_75.append(convergence_times_for_count_optimal_75)
    #convergence_times_optimal.append(convergence_times_for_count_optimal)
   # convergence_times_optimal_25.append(convergence_times_for_count_optimal_25)
# Calculate and print the average degree distribution for each connection type
"""
if total_nodes_nnc > 0:
    average_degree_nnc = total_degrees_nnc / total_nodes_nnc
    print(f"Average Degree Distribution for NNC: {average_degree_nnc}")
else:
    print("No nodes found in the NNC graph.")
"""
if total_nodes_random > 0:
    average_degree_random = total_degrees_random / total_nodes_random
    print(f"Average Degree Distribution for Random: {average_degree_random}")
else:
    print("No nodes found in the Random graph.")



"""
if total_nodes_optimal > 0:
    average_degree_optimal = total_degrees_optimal / total_nodes_optimal
    print(f"Average Degree Distribution for PDBCA with alpha 0.5 and beta 0.5: {average_degree_optimal}")
else:
    print("No nodes found in the Degree balanced NNC graph.")
"""

# Print final results
#print("Convergence Times for NNC:", convergence_times)
print("Convergence Times for Random Connection:", convergence_times_random)
#print("Convergence Times for Nearest Neighbor with Degree Limit:", convergence_times_nearest_limit)
#print("Convergence Times for PDBCA with alpha 0.75 and beta 0.25:", convergence_times_optimal_75)
#print("Convergence Times for  PDBCA with alpha 0.5 and beta 0.5:", convergence_times_optimal)
#print("Convergence Times for  PDBCA with alpha 0.25 and beta 0.75:", convergence_times_optimal_25)
print("################################################################")
#print("Total Messages forwarded for NNC:", message_NNC)
print("Total Messages forwarded for Random Connection:", message_random)
#print("Convergence Times for Nearest Neighbor with Degree Limit:", convergence_times_nearest_limit)
#print("Total Messages forwarded for PDBCA with alpha 0.75 and beta 0.25:", message_PDBCA_75)
#print("Total Messages forwarded  for  PDBCA with alpha 0.5 and beta 0.5:", message_PDBCA_50)
#print("Total Messages forwarded  for  PDBCA with alpha 0.25 and beta 0.75:", message_PDBCA_25)












2000 number of nodes done
4000 number of nodes done
6000 number of nodes done
8000 number of nodes done
10000 number of nodes done
--- Progress: 10000 nodes ---
Time for this 10k block: 0.07 minutes
Total time elapsed: 0.07 minutes
-------------------------
12000 number of nodes done
14000 number of nodes done
16000 number of nodes done
18000 number of nodes done
20000 number of nodes done
--- Progress: 20000 nodes ---
Time for this 10k block: 0.07 minutes
Total time elapsed: 0.14 minutes
-------------------------
22000 number of nodes done
24000 number of nodes done
26000 number of nodes done
28000 number of nodes done
30000 number of nodes done
--- Progress: 30000 nodes ---
Time for this 10k block: 0.08 minutes
Total time elapsed: 0.22 minutes
-------------------------
Random neigbour connection
number of nodes29986 
Degree counts saved to Random_50000_4_rmin.csv
Total messages forwarded: 449616
Total messages forwarded: 449616
Total messages forwarded: 449616
Total messages forwarde